<a href="https://colab.research.google.com/github/linoy25/Project-OnlyPlants/blob/main/Tutorials/tirgul7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
from nltk.stem import PorterStemmer

In [ ]:
# ==========================================
# 1. Index Service - שירות האינדוקס
# ==========================================
class IndexService:
    def __init__(self):
        self.documents = {} # שומר את התוכן המקורי של כל מסמך
        self.index = {}     # האינדקס ההפוך: {מילה_אחרי_stemming: {מזהה_מסמך: תדירות}}
        self.stemmer = PorterStemmer()
        self.stop_words = {'a', 'an', 'the', 'and', 'or', 'in', 'on', 'at'}

    def add_document(self, doc_id, title, text):
        self.documents[doc_id] = {'title': title, 'content': text}
        words = re.findall(r'\w+', text.lower())

        # שלב 1 (מתרגול 5): הסרת מילות קישור וביצוע Stemming
        for word in words:
            if word not in self.stop_words:
                stemmed_word = self.stemmer.stem(word)

                # שלב 2: בניית אינדקס שומר תדירויות (לצורך דירוג)
                if stemmed_word not in self.index:
                    self.index[stemmed_word] = {}
                if doc_id not in self.index[stemmed_word]:
                    self.index[stemmed_word][doc_id] = 0
                self.index[stemmed_word][doc_id] += 1

    def get_document(self, doc_id):
        return self.documents.get(doc_id)

In [ ]:
# ==========================================
# 2. Query Service - שירות השאילתות
# ==========================================
class QueryService:
    def __init__(self, index_service):
        self.index_service = index_service
        self.stemmer = PorterStemmer()
        self.queries = {}

    def create_query(self, query_data):
        query_id = str(len(self.queries) + 1)
        raw_terms = query_data.get('terms', [])
        operator = query_data.get('operator', 'OR').upper() # ברירת מחדל היא OR

        # ביצוע Stemming למילות החיפוש (כדי שיתאימו לאינדקס)
        search_terms = [self.stemmer.stem(term.lower()) for term in raw_terms]

        # מציאת מסמכים עבור כל מילת חיפוש
        term_docs = []
        for term in search_terms:
            docs_with_term = set(self.index_service.index.get(term, {}).keys())
            term_docs.append(docs_with_term)

        # --- תמיכה באופרטורים לוגיים (AND, OR) ---
        if not term_docs:
            final_docs = set()
        elif operator == 'AND':
            final_docs = set.intersection(*term_docs) if term_docs else set()
        else: # OR
            final_docs = set.union(*term_docs) if term_docs else set()

        # --- מנגנון דירוג (Ranking) מתרגול 5 ---
        doc_scores = {}
        for doc_id in final_docs:
            score = 0
            for term in search_terms:
                # ככל שהמילה מופיעה יותר פעמים במסמך, הציון עולה
                freq = self.index_service.index.get(term, {}).get(doc_id, 0)
                score += freq
            doc_scores[doc_id] = score

        # מיון התוצאות מהציון הגבוה לנמוך
        ranked_results = sorted(list(final_docs), key=lambda x: doc_scores[x], reverse=True)

        query_record = {
            'id': query_id,
            'terms': raw_terms,
            'operator': operator,
            'results': ranked_results,
            'scores': {doc: doc_scores[doc] for doc in ranked_results}
        }
        self.queries[query_id] = query_record
        return query_record

In [ ]:
# ==========================================
# 3. Result Service - שירות התוצאות
# ==========================================
class ResultService:
    def __init__(self, index_service, query_service):
        self.index_service = index_service
        self.query_service = query_service

    def format_results(self, query_id):
        query = self.query_service.queries.get(query_id)
        if not query:
            return {'error': 'Query not found'}

        formatted = []
        for doc_id in query['results']:
            doc = self.index_service.get_document(doc_id)
            score = query['scores'][doc_id]

            formatted.append({
                'doc_id': doc_id,
                'title': doc['title'],
                'score': score, # מציג את הדירוג
                'snippet': doc['content'][:60] + '...'
            })

        return {
            'query_id': query_id,
            'operator': query['operator'],
            'total_results': len(formatted),
            'results': formatted
        }

In [ ]:
# ==========================================
# הדגמת הרצה לפרויקט OnlyPlants! 🌱
# ==========================================
if __name__ == "__main__":
    # אתחול השירותים
    index_service = IndexService()
    query_service = QueryService(index_service)
    result_service = ResultService(index_service, query_service)

    # הוספת מסמכים לדוגמה (מבוסס על פרויקט הסחלבים שלכם)
    index_service.add_document('1', 'Orchid Disease Detection',
                               'The sensor detects leaf disease in the orchid. The disease spreads fast on the leaf.')
    index_service.add_document('2', 'Moisture Monitor',
                               'Moisture sensor checks the water level of the orchid root.')
    index_service.add_document('3', 'Light and Temperature',
                               'Temperature and light are important for health. A light sensor helps.')

    # חיפוש 1: אופרטור OR (יחזיר מסמכים שיש בהם orchid או disease) + בדיקת דירוג
    print("--- Search 1: OR Operator (orchid, disease) ---")
    q1 = query_service.create_query({'terms': ['orchid', 'disease'], 'operator': 'OR'})
    res1 = result_service.format_results(q1['id'])
    for r in res1['results']:
        print(f"[{r['doc_id']}] {r['title']} | Score: {r['score']} | {r['snippet']}")

    # חיפוש 2: אופרטור AND (יחזיר רק מסמכים שיש בהם גם orchid וגם sensor)
    print("\n--- Search 2: AND Operator (orchid, sensor) ---")
    q2 = query_service.create_query({'terms': ['orchid', 'sensor'], 'operator': 'AND'})
    res2 = result_service.format_results(q2['id'])
    for r in res2['results']:
        print(f"[{r['doc_id']}] {r['title']} | Score: {r['score']} | {r['snippet']}")